# 01 · Data — anonymized operations of a 2-vehicle ride-hailing fleet

**Source:** the owner's daily operations workbooks for **Vemov** (Luanda, Angola), built from the Yango/Heetch driver apps, fuel receipts, odometer readings and invoices. They are private and stay on the owner's machine.

[`src/extract_anonymized.py`](../src/extract_anonymized.py) turns them into the six tables used here:
- Vehicles, drivers and platforms are replaced by IDs (V01, D01, ...).
- **Not exported:** names, plates, phone numbers, pick-up locations, fuel stations or free-text notes.
- Notes are mapped to neutral stop categories.
- **Money is indexed:** every amount is real kwanza × one confidential factor. Ratios, percentages, payback months, trips, hours and km are exact; absolute amounts are not. Fuel litres are not published, because with the public fuel price they would reveal the factor.

| Table | Grain | Main columns |
|---|---|---|
| `daily` | vehicle × day | status, trips, gross fares, platform fees, bonus, net revenue, fuel cost, first/last trip |
| `trips` | trip | date, start time, net value |
| `costs` | cost event | maintenance, driver salary, wash, parking, equipment; documented yes/no |
| `odometer` | reading | vehicle, date, km |
| `vehicles` | vehicle | acquisition value and date, financing, insurance |
| `drivers` | driver | vehicle, platform, settlement model, first/last day |

**Day status:**

| Status | Meaning |
|---|---|
| `operated` | The vehicle worked that day |
| `rest_day` | Contractual weekly rest |
| `driver_absence` | Driver absent |
| `no_driver` | No driver assigned (between contracts) |
| `network_outage` | The service phone had no network, so the vehicle could not take trips |

In [1]:
import re
from pathlib import Path

import duckdb
import pandas as pd

DATA = Path("../data/anonymized")
TABLES = ["daily", "trips", "costs", "odometer", "vehicles", "drivers"]

con = duckdb.connect()
for t in TABLES:
    con.execute(f"CREATE VIEW {t} AS SELECT * FROM read_csv_auto('{(DATA / (t + '.csv')).as_posix()}')")

sql_text = Path("../sql/kpis.sql").read_text(encoding="utf-8")
QUERIES = dict(re.findall(r"-- name: (\w+)\n(.*?)(?=\n-- name:|\Z)", sql_text, re.S))

def run(name: str) -> pd.DataFrame:
    return con.execute(QUERIES[name]).df()

## 1 · Coverage

In [2]:
con.execute("""
    SELECT vehicle_id, MIN(date) AS first_day, MAX(date) AS last_day, COUNT(*) AS days,
           SUM(trips) AS trips, ROUND(SUM(net_revenue_aoa)) AS net_revenue_aoa
    FROM daily GROUP BY vehicle_id ORDER BY vehicle_id""").df()

,vehicle_id,first_day,last_day,days,trips,net_revenue_aoa
0,V01,2026-07-01,2026-10-04,96,1058.0,2446972.0
1,V02,2026-08-18,2026-09-22,36,186.0,348049.0


In [3]:
con.execute("SELECT * FROM drivers").df()

,driver_id,vehicle_id,platform,first_day,last_day,settlement_model,weekly_minimum_aoa
0,D01,V01,Yango,2026-07-01,2026-10-03,weekly_minimum,109639.2
1,D02,V02,Heetch,2026-08-18,2026-08-28,full_remittance,NaN
2,D03,V02,Yango,2026-09-16,2026-09-22,weekly_minimum,109639.2


In [4]:
pd.crosstab(con.execute("SELECT vehicle_id, status FROM daily").df()["vehicle_id"],
            con.execute("SELECT vehicle_id, status FROM daily").df()["status"])

status,driver_absence,network_outage,no_driver,operated,rest_day
vehicle_id,,,,,
V01,15,1,4,65,11
V02,6,0,10,17,3


## 2 · Reconciliation

**With the source workbooks.** The extraction script stops if trips or net revenue differ from the TOTAL row of any workbook, before anything is indexed. Run log:

```
✅ D01: trips and net revenue match the workbook TOTAL row
✅ D02: trips and net revenue match the workbook TOTAL row
✅ D03: trips and net revenue match the workbook TOTAL row
```

**Inside the published tables.** Trip-level detail must agree with the daily totals.

In [5]:
# Trip-level detail covers almost every daily total (a few screenshots cut the trip time off)
con.execute("""
    WITH t AS (SELECT driver_id, COUNT(*) AS trips_detail, SUM(net_value_aoa) AS net_detail FROM trips GROUP BY 1),
         d AS (SELECT driver_id, SUM(trips) AS trips_daily, SUM(net_revenue_aoa) AS net_daily FROM daily
               WHERE driver_id IS NOT NULL GROUP BY 1)
    SELECT driver_id, trips_detail, trips_daily, ROUND(100.0 * trips_detail / trips_daily, 1) AS pct_trips_with_detail
    FROM t JOIN d USING (driver_id) ORDER BY 1""").df()

,driver_id,trips_detail,trips_daily,pct_trips_with_detail
0,D01,1050,1058.0,99.2
1,D02,84,84.0,100.0
2,D03,102,102.0,100.0


## 3 · Known limits of the data

| # | Limit | Consequence |
|---|---|---|
| 1 | Wash (V01) and parking have **no receipts**; they are posted from the owner's weekly routine | Costs marked `documented = False` |
| 2 | **Parking** is booked on every calendar day at the weekly rate, because it is paid even when the car is idle | ~23% more parking than the owner's workbook, which skipped idle days |
| 3 | Two of V01's three services and V02's first service have no invoice | Amounts taken from the owner's records, `documented = False` |
| 4 | D03's first week has almost no fuel receipts | Fuel cost per km for V02 is not reliable after 16 Sep |
| 5 | V02 has no workbook between drivers (4–15 Sep) | Those days are added as `no_driver` (Sundays as `rest_day`) |
| 6 | No insurance policy is documented for V01 | V01 margin is before insurance |

**Next:** `02_analysis.ipynb`.